<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
#!pip install pandas
#!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [2]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

In [3]:
import numpy as np

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [4]:
missing_values = df.isnull()
df_columns = df.columns.values.tolist()
for column in df_columns:
    print(column)
    print('type: ', df[column].dtypes, '  count: ', df[column].count(), '  missing values:', missing_values[column].sum(), '  % of missing values: ', 
          f'{(missing_values[column].sum() / len(df.index)) * 100: .2f}%', '\n')

ResponseId
type:  int64   count:  65437   missing values: 0   % of missing values:   0.00% 

MainBranch
type:  object   count:  65437   missing values: 0   % of missing values:   0.00% 

Age
type:  object   count:  65437   missing values: 0   % of missing values:   0.00% 

Employment
type:  object   count:  65437   missing values: 0   % of missing values:   0.00% 

RemoteWork
type:  object   count:  54806   missing values: 10631   % of missing values:   16.25% 

Check
type:  object   count:  65437   missing values: 0   % of missing values:   0.00% 

CodingActivities
type:  object   count:  54466   missing values: 10971   % of missing values:   16.77% 

EdLevel
type:  object   count:  60784   missing values: 4653   % of missing values:   7.11% 

LearnCode
type:  object   count:  60488   missing values: 4949   % of missing values:   7.56% 

LearnCodeOnline
type:  object   count:  49237   missing values: 16200   % of missing values:   24.76% 

TechDoc
type:  object   count:  40897   missi

<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [5]:
df.describe()

,ResponseId,CompTotal,WorkExp,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,ConvertedCompYearly,JobSat
count,65437.000000,3.374000e+04,29658.000000,29324.000000,29393.000000,29411.000000,29450.000000,29448.00000,29456.000000,29456.000000,29450.000000,29445.000000,2.343500e+04,29126.000000
mean,32719.000000,2.963841e+145,11.466957,18.581094,7.522140,10.060857,24.343232,22.96522,20.278165,16.169432,10.955713,9.953948,8.615529e+04,6.935041
std,18890.179119,5.444117e+147,9.168709,25.966221,18.422661,21.833836,27.089360,27.01774,26.108110,24.845032,22.906263,21.775652,1.867570e+05,2.088259
min,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000e+00,0.000000
25%,16360.000000,6.000000e+04,4.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,3.271200e+04,6.000000
50%,32719.000000,1.100000e+05,9.000000,10.000000,0.000000,0.000000,20.000000,15.00000,10.000000,5.000000,0.000000,0.000000,6.500000e+04,7.000000
75%,49078.000000,2.500000e+05,16.000000,22.000000,5.000000,10.000000,30.000000,30.00000,25.000000,20.000000,10.000000,10.000000,1.079715e+05,8.000000
max,65437.000000,1.000000e+150,50.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,1.625660e+07,10.000000


### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [6]:
pd.set_option('display.max_rows', 200)

In [7]:
len(df['Country'].value_counts().values.tolist())

185

In [8]:
#let's look at the unique values for the Country column
df['Country'].value_counts().sort_index().head(200)

Country
Afghanistan                                                56
Albania                                                    49
Algeria                                                    77
Andorra                                                    15
Angola                                                     20
Antigua and Barbuda                                         5
Argentina                                                 345
Armenia                                                    58
Australia                                                1260
Austria                                                   791
Azerbaijan                                                 27
Bahamas                                                     4
Bahrain                                                    11
Bangladesh                                                327
Barbados                                                    6
Belarus                                                    97


###Examining the list of unique values we see that we have 'North Korea' and 'Democratic People's Republic of Korea', which are the same. The same happens with 'South Korea'and 'Republic of Korea'.

In [9]:
#Let's now have a look at EdLevel
df['EdLevel'].value_counts().sort_index().head(200)

EdLevel
Associate degree (A.A., A.S., etc.)                                                    1793
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          24942
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       15557
Primary/elementary school                                                              1146
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         2970
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     5793
Some college/university study without earning a degree                                 7651
Something else                                                                          932
Name: count, dtype: int64

###It seems to me that we don't have irrelevant or inconsistent values in this column...

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [10]:
df['Country'] = df['Country'].replace('North Korea', "Democratic People's Republic of Korea")
df['Country'] = df['Country'].replace('South Korea', "Republic of Korea")
df['Country'].value_counts().sort_index().head(200)

Country
Afghanistan                                                56
Albania                                                    49
Algeria                                                    77
Andorra                                                    15
Angola                                                     20
Antigua and Barbuda                                         5
Argentina                                                 345
Armenia                                                    58
Australia                                                1260
Austria                                                   791
Azerbaijan                                                 27
Bahamas                                                     4
Bahrain                                                    11
Bangladesh                                                327
Barbados                                                    6
Belarus                                                    97


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [11]:
#let's first have a look on the unique values for this column
print(len(df['Country'].value_counts().values.tolist()))
df['Employment'].value_counts().head(200)

183


Employment
Employed, full-time                                                                                                                                                                                                     39041
Independent contractor, freelancer, or self-employed                                                                                                                                                                     4846
Student, full-time                                                                                                                                                                                                       4709
Employed, full-time;Independent contractor, freelancer, or self-employed                                                                                                                                                 3557
Not employed, but looking for work                                                                   

In [12]:
#let's determine the original unique options in the survey
all_options = df['Employment'].dropna().str.split(';').explode().unique()
print(all_options)

['Employed, full-time' 'Student, full-time'
 'Not employed, but looking for work'
 'Independent contractor, freelancer, or self-employed'
 'Not employed, and not looking for work' 'Student, part-time'
 'Employed, part-time' 'I prefer not to say' 'Retired']


In [13]:
#let's encode creating a boolean column for each original unique value 
for et in all_options:
    df[et] = df['Employment'].str.contains(et, na=False)
df.head()

,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSat,"Employed, full-time","Student, full-time","Not employed, but looking for work","Independent contractor, freelancer, or self-employed","Not employed, and not looking for work","Student, part-time","Employed, part-time",I prefer not to say,Retired
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,NaN,True,False,False,False,False,False,False,False,False
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,True,False,False,False,False,False,False,False,False
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,True,False,False,False,False,False,False,False,False
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,NaN,False,True,False,False,False,False,False,False,False
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,NaN,False,True,False,False,False,False,False,False,False


### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [14]:
missing_values = df.isnull()
missing_values.sum().sort_values(ascending=False)

AINextMuch less integrated                              64289
AINextLess integrated                                   63082
AINextNo change                                         52939
AINextMuch more integrated                              51999
EmbeddedAdmired                                         48704
EmbeddedWantToWorkWith                                  47837
EmbeddedHaveWorkedWith                                  43223
ConvertedCompYearly                                     42002
AIToolNot interested in Using                           41023
AINextMore integrated                                   41009
Knowledge_9                                             37802
Frequency_3                                             37727
Knowledge_8                                             37679
ProfessionalTech                                        37673
Knowledge_7                                             37659
Knowledge_6                                             37573
Knowledg

<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [15]:
#let's replace by the median
CCY_median = df['ConvertedCompYearly'].median()
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].replace(np.nan, CCY_median)
df['ConvertedCompYearly'].isnull().sum()

np.int64(0)

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [16]:
#let's see which is the most frequent value for RemoteWork
df['RemoteWork'].value_counts()

RemoteWork
Hybrid (some remote, some in-person)    23015
Remote                                  20831
In-person                               10960
Name: count, dtype: int64

In [17]:
#impute most frequent value to RemoteWork
df['RemoteWork'] = df['RemoteWork'].replace(np.nan, 'Hybrid (some remote, some in-person)')
df['RemoteWork'].isnull().sum()

np.int64(0)

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [18]:
#let's create a new column with ConvertedCompYearly normalized; we want to preserve the original column to deal with it later
comp_range = df['ConvertedCompYearly'].max() - df['ConvertedCompYearly'].min()
df['ConvertedCompYearly_MinMax'] = (df['ConvertedCompYearly'] - df['ConvertedCompYearly'].min() ) / comp_range
df[['ConvertedCompYearly_MinMax']].head()

,ConvertedCompYearly_MinMax
0,0.003998
1,0.003998
2,0.003998
3,0.003998
4,0.003998


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [19]:
## We will use natural logarithm and create a new column
df['ConvertedCompYearly_log'] = np.log(df['ConvertedCompYearly'])
df[['ConvertedCompYearly_log']].head()

,ConvertedCompYearly_log
0,11.082143
1,11.082143
2,11.082143
3,11.082143
4,11.082143


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [20]:
#let's first see if we have missing values in the YearsCodePro column
df['YearsCodePro'].isnull().sum()

np.int64(13827)

In [21]:
#we will not deal with the missing values here, we will preserve them to be analyzed later...
df['YearsCodePro'].unique()

array([nan, '17', '27', '7', '11', '25', '12', '10', '3',
       'Less than 1 year', '18', '37', '15', '20', '6', '2', '16', '8',
       '14', '4', '45', '1', '24', '29', '5', '30', '26', '9', '33', '13',
       '35', '23', '22', '31', '19', '21', '28', '34', '32', '40', '50',
       '39', '44', '42', '41', '36', '38', 'More than 50 years', '43',
       '47', '48', '46', '49'], dtype=object)

In [22]:
#lets define a function to assign the experience levels
def exp_level(code_years_pro):
    exp_levels = {1: 'Trainee', 6: 'Junior', 11: 'Full', 21: 'Senior', 35: 'Expert', 100: 'Consultant'}

    if pd.isna(code_years_pro):
        return np.nan
    if code_years_pro == 'Less than 1 year':
        return 'Trainee'
    if code_years_pro == 'More than 50 years':
        return 'Consultant'

    try:
        years = float(code_years_pro)
    except (ValueError, TypeError):
        return np.nan

    for limit, exp in exp_levels.items():
        if years < limit:
            return exp

In [23]:
#let's create the new column
df['ExperienceLevel'] = df['YearsCodePro']. apply(exp_level)
df[['YearsCodePro', 'ExperienceLevel']].head()

,YearsCodePro,ExperienceLevel
0,NaN,NaN
1,17,Senior
2,27,Expert
3,NaN,NaN
4,NaN,NaN


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
